Imports and paths

In [3]:
import numpy as np
import pandas as pd

DATA_DIR = "../data/processed"
TF = "1h"  # change to 4h, 1d, etc. if needed

prefix = f"btc_{TF}_featured_"

X_path = f"{DATA_DIR}/{prefix}labels_X.npy"
y_path = f"{DATA_DIR}/{prefix}labels_y.npy"
t_path = f"{DATA_DIR}/{prefix}labels_t.npy"
label_cols_path = f"{DATA_DIR}/{prefix}label_cols.npy"


Load arrays

In [4]:
X = np.load(X_path)
y = np.load(y_path)
t = np.load(t_path)
label_cols = np.load(label_cols_path)

print("X shape:", X.shape)      # (num_windows, window_size, num_features)
print("y shape:", y.shape)      # (num_windows, num_horizons)
print("t shape:", t.shape)
print("label_cols:", label_cols)



X shape: (54131, 100, 21)
y shape: (54131, 2)
t shape: (54131,)
label_cols: ['label_1h' 'label_4h']


In [5]:
import pandas as pd

i = 0
print("Window time:", pd.to_datetime(t[i]))
print("Labels:", dict(zip(label_cols, y[i])))


Window time: 2020-01-07 04:00:00
Labels: {'label_1h': 0, 'label_4h': 0}


Inspect label distributions

In [6]:
for i, col in enumerate(label_cols):
    vals, counts = np.unique(y[:, i], return_counts=True)
    dist = dict(zip(vals, counts))
    print(f"{col} distribution:", dist)


label_1h distribution: {-2: 553, -1: 2453, 0: 47983, 1: 2555, 2: 587}
label_4h distribution: {-2: 4266, -1: 2491, 0: 40086, 1: 2738, 2: 4550}


Quick sanity check on one sample

In [10]:
idx = 19  # pick a window index
print("Timestamp:", pd.to_datetime(t[idx]))
print("Labels:", {label_cols[i]: int(y[idx, i]) for i in range(len(label_cols))})

# Show last few timesteps of close + returns for context
df_sample = pd.DataFrame(
    X[idx],
    columns=[
        "open", "high", "low", "close", "volume",
        "returns", "hl_range", "hl_pct",
        "volatility", "volatility_short",
        "sma_20", "sma_50", "sma_ratio",
        "rsi", "macd", "macd_signal", "macd_histogram",
        "bb_width", "bb_position",
        "volume_ratio", "price_position",
    ],
)
df_sample.tail()


Timestamp: 2020-01-07 23:00:00
Labels: {'label_1h': 2, 'label_4h': 2}


,open,high,low,close,volume,returns,hl_range,hl_pct,volatility,volatility_short,...,sma_50,sma_ratio,rsi,macd,macd_signal,macd_histogram,bb_width,bb_position,volume_ratio,price_position
95,8038.540039,8111.450195,8032.390137,8073.689941,4283.208496,0.004479,79.059998,0.979230,0.723526,1.002945,...,7667.201172,1.029441,74.034050,86.690804,79.568901,7.121901,3.200394,1.199571,1.148645,0.922698
96,8073.689941,8175.009766,8041.279785,8162.660156,4404.727539,0.011020,133.729996,1.638314,0.741514,0.981569,...,7681.680176,1.029285,77.823730,99.964806,83.648087,16.316725,4.278095,1.233148,1.329360,0.985589
97,8162.569824,8188.000000,7975.740234,8016.439941,5116.557617,-0.017913,212.259995,2.647809,0.858263,1.277617,...,7693.125977,1.028474,61.847675,97.561195,86.430710,11.130489,4.516656,0.787955,1.513767,0.802805
98,8016.430176,8067.060059,7945.720215,8049.549805,3088.533447,0.004130,121.339996,1.507414,0.795252,1.091574,...,7705.484863,1.027819,63.666542,97.207466,88.586060,8.621409,4.731377,0.840560,0.907987,0.840862
99,8049.830078,8207.679688,8020.700195,8145.279785,3983.395020,0.011893,186.979996,2.295563,0.817767,1.152203,...,7721.020020,1.027373,68.362801,103.459137,91.560677,11.898461,5.247647,0.998122,1.134834,0.929862


Find an index where label_1h_5c == 2

In [8]:
import numpy as np

# Find column index for label_1h_5c
label_name = "label_1h"
label_idx = list(label_cols).index(label_name)

# Indices of windows where label_1h_5c == 2 (Bullish)
bull_idxs = np.where(y[:, label_idx] == 2)[0]
print("Found", len(bull_idxs), "bullish continuation windows")

# Pick one example index (e.g. the first one)
if len(bull_idxs) > 0:
    i = int(bull_idxs[0])
    print("Inspecting window index:", i)
else:
    i = None


Found 587 bullish continuation windows
Inspecting window index: 19


Inspect that window (if found)

In [9]:
import pandas as pd

if i is not None:
    print("Timestamp:", pd.to_datetime(t[i]))
    print("Labels:", {label_cols[j]: int(y[i, j]) for j in range(len(label_cols))})

    df_sample = pd.DataFrame(
        X[i],
        columns=[
            "open", "high", "low", "close", "volume",
            "returns", "hl_range", "hl_pct",
            "volatility", "volatility_short",
            "sma_20", "sma_50", "sma_ratio",
            "rsi", "macd", "macd_signal", "macd_histogram",
            "bb_width", "bb_position",
            "volume_ratio", "price_position",
        ],
    )
    df_sample.tail(10)  # last 10 candles of the window


Timestamp: 2020-01-07 23:00:00
Labels: {'label_1h': 2, 'label_4h': 2}
